# Optunaによるパラメータのオートチューニング

In [ ]:
!pip install -qq optuna kaggle-environments

In [ ]:
"""Colabで実行するパラメータ探索コード。"""

import importlib
import statistics

import optuna
from kaggle_environments import make

import main as strategy


# Colab上の最新のmain.pyを読み込む。
strategy = importlib.reload(strategy)

N_TRIALS = 100
MATCH_COUNT = 20
EVALUATION_SEEDS = list(range(MATCH_COUNT))


def objective(trial):
    """平均得点を返す。"""

    #メロンの種数と植付済み数を合わせた購入目標数
    strategy.StrategyConfig.MELON_TARGET_COUNT = trial.suggest_int(
        "MELON_TARGET_COUNT",
        4,
        16,
        step=2,
    )

    #メロンの植付けと種購入を許可する期限
    strategy.StrategyConfig.MELON_PLANT_END_DAY = trial.suggest_int(
        "MELON_PLANT_END_DAY",
        4,
        10,

    )

    rewards = []

    for episode_seed in EVALUATION_SEEDS:
        strategy.hire_controller = strategy.HireController()

        env = make(
            "kaggriculture",
            configuration={
                "episodeSteps": 720,
                "seed": episode_seed,
            },
            debug=True,
        )

        env.run([strategy.agent, strategy.agent])

        for state in env.steps[-1]:
            rewards.append(float(state.reward))

    return statistics.fmean(rewards)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)

print("\n===== 最高平均得点 =====")
print(study.best_value)

print("\n===== main.pyへ手動設定する値 =====")

for name, value in study.best_params.items():
    print(f"{name} = {value}")

[I 2026-09-09 01:05:49,106] A new study created in memory with name: no-name-ce245004-23c1-4565-9aea-5ba5ea012696


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-09-09 01:09:14,999] Trial 0 finished with value: 60588.75 and parameters: {'MELON_TARGET_COUNT': 8, 'MELON_PLANT_END_DAY': 10}. Best is trial 0 with value: 60588.75.
[I 2026-09-09 01:13:13,392] Trial 1 finished with value: 55502.125 and parameters: {'MELON_TARGET_COUNT': 14, 'MELON_PLANT_END_DAY': 8}. Best is trial 0 with value: 60588.75.
[I 2026-09-09 01:16:39,884] Trial 2 finished with value: 56510.1 and parameters: {'MELON_TARGET_COUNT': 6, 'MELON_PLANT_END_DAY': 5}. Best is trial 0 with value: 60588.75.
[I 2026-09-09 01:20:00,533] Trial 3 finished with value: 56922.05 and parameters: {'MELON_TARGET_COUNT': 4, 'MELON_PLANT_END_DAY': 10}. Best is trial 0 with value: 60588.75.
[I 2026-09-09 01:23:20,744] Trial 4 finished with value: 64378.375 and parameters: {'MELON_TARGET_COUNT': 12, 'MELON_PLANT_END_DAY': 8}. Best is trial 4 with value: 64378.375.
[I 2026-09-09 01:26:43,642] Trial 5 finished with value: 56922.05 and parameters: {'MELON_TARGET_COUNT': 4, 'MELON_PLANT_END_DAY'